# Spoken-Word Recognition: the full ML cycle

**Problem:** voice-driven tools (hands-free devices, accessibility tools, learning apps) must recognise short spoken
words. This project classifies 1-second recordings of 8 spoken words (*yes, no, up, down, left, right, stop, go*)
from the public **Google Speech Commands** dataset, so an app can respond to a person's voice.
It is a proof of concept: the same pipeline can be retrained on recordings in another language
(for example Kinyarwanda) by changing only the data.

**Data type:** sound (.wav). **Model:** MobileNetV2 transfer learning on log-mel spectrograms,
compared with a vanilla CNN. The same code lives in `src/` and powers the API and UI.

Sections: 1 setup, 2 data acquisition, 3 exploration + interpretations, 4 preprocessing, 5 baseline,
6 optimised model, 7 evaluation, 8 prediction, 9 retraining demo, 10 conclusion.

## 1. Setup

In [ ]:
import os, sys, random, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/spoken-word-mlops"):
        os.system("git clone https://github.com/Queenlinda12/spoken-word-mlops.git /content/spoken-word-mlops")
    os.system("pip install -q librosa soundfile scikit-learn seaborn")
    os.chdir("/content/spoken-word-mlops")
elif os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")
sys.path.append(os.getcwd())
if not os.path.isdir("raw_recordings"):              # first run: download Google Speech Commands (~2.3 GB) and build the folders
    os.system(f"{sys.executable} scripts/prepare_speech_commands.py")

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, librosa, librosa.display
import tensorflow as tf
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report, roc_auc_score
from src import preprocessing as pp, model as mdl, prediction

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
EPOCHS = int(os.getenv("NB_EPOCHS", 15))
FT_EPOCHS = int(os.getenv("NB_FT_EPOCHS", 10))
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Data acquisition
The data is the public **Google Speech Commands v0.02** dataset (Warden, 2018, CC BY 4.0): thousands of people saying
short words. `scripts/prepare_speech_commands.py` keeps 8 words and builds `raw_recordings/<word>/<speaker>__<n>.wav`
(40 speakers) plus `new_recordings/` (6 *different* speakers, used later for the retraining demo).
`split_raw_dataset` then builds `data/train` and `data/test`. With at least 4 speakers it **holds out whole speakers
for the test set**, so the test score measures how well the model understands a *new voice*, which is what matters in real use.

In [ ]:
from pathlib import Path
RAW_DIR = os.getenv("RAW_DIR", "raw_recordings")     # <- one folder per word
assert os.path.isdir(RAW_DIR), f"'{RAW_DIR}' not found. Run scripts/prepare_speech_commands.py first"
pp.DB_PATH.unlink(missing_ok=True)                    # fresh uploads database for a clean run
summary = pp.split_raw_dataset(RAW_DIR, test_ratio=0.2, seed=SEED)
print("split mode:", summary["mode"], "| test speakers (never used for training):", summary["test_speakers"])
display(pd.DataFrame(summary["classes"]).T)
raw_tbl = pd.crosstab([c for c in sorted(summary["classes"]) for f in pp.list_audio(Path(RAW_DIR) / c)],
                      [pp.speaker_of(f) for c in sorted(summary["classes"]) for f in pp.list_audio(Path(RAW_DIR) / c)])
print("clips per word x speaker:"); display(raw_tbl)

### My dataset (fill in)
* Source: Google Speech Commands v0.02 (public, CC BY 4.0). Words used: ______________________________________
* Number of speakers: ______ main + ______ new | Total clips: ______ | Clips per word: ______ to ______
* What kind of recordings are these (length, sampling rate, background noise, many different microphones)? ______________
* Speakers held out as the test set: ______ (why this matters: ______________________________________)

## 3. Exploratory data analysis: what story does the data tell?

In [ ]:
classes = pp.get_class_names()
ins = pp.dataset_insights(max_per_class=100)
# Feature 1: class balance + speakers
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
sns.barplot(x=list(ins["counts"]), y=list(ins["counts"].values()), ax=ax[0]); ax[0].set_title("Clips per word"); ax[0].tick_params(axis="x", rotation=40)
sns.barplot(x=list(ins["speakers"]), y=list(ins["speakers"].values()), ax=ax[1]); ax[1].set_title("Distinct speakers per word"); ax[1].tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()

# Features 2-4: duration, spectral centroid, loudness
rows = [(c, f, v) for f, d in ins["features"].items() for c, vals in d.items() for v in vals]
df = pd.DataFrame(rows, columns=["word", "feature", "value"])
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for ax, f in zip(axes, ["voiced_duration", "spectral_centroid", "loudness_db"]):
    sns.boxplot(data=df[df.feature == f], x="word", y="value", ax=ax); ax.set_title(f); ax.tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()
from IPython.display import Markdown
Markdown("\n\n".join("- " + n for n in ins["interpretations"]))

In [ ]:
# One waveform + spectrogram per word
fig, axes = plt.subplots(2, len(classes), figsize=(2.6 * len(classes), 4.5))
for i, c in enumerate(classes):
    f = random.choice(pp.list_audio(pp.TRAIN_DIR / c))
    y = pp.trim_and_fit(pp.load_waveform(f))
    axes[0][i].plot(y, lw=0.5); axes[0][i].set_title(c); axes[0][i].axis("off")
    axes[1][i].imshow(pp.waveform_to_spec(y)[:, :, 0], origin="lower", aspect="auto", cmap="magma"); axes[1][i].axis("off")
plt.suptitle("Waveform (top) and log-mel spectrogram (bottom)"); plt.show()

### Interpretation of the data (fill in after looking at the plots above: 3 features minimum)
Write what you SEE, then what it MEANS.

1. **Class balance and speakers** (plot: clips and speakers per word)
   * I see: ________________________________________________________________
   * It means / what I do about it: ________________________________________________

2. **Spoken duration** (box plot)
   * I see (which words are longest/shortest, how big is the spread): __________________________
   * It means (syllables? speaking speed?): _________________________________________

3. **Spectral centroid, the "brightness" of the sound** (box plot)
   * I see: ________________________________________________________________
   * It means (which sounds carry high frequencies, do classes overlap?): _____________________

4. **Loudness** (box plot)
   * I see: ________________________________________________________________
   * It means (microphone distance? device?) and why we normalise it: _______________________

5. **Spectrograms** (waveform + spectrogram per word)
   * Which two words look most alike, and which look most different: ________________________
   * What this predicts about which words the model will confuse: ___________________________

## 4. Preprocessing
| Step | Why |
|---|---|
| Read audio, convert to mono, resample to 16 kHz | one consistent format for every phone / microphone |
| Trim leading/trailing silence, centre the word in a 2 s window | recordings start at random moments; fixed-size input |
| 128-band log-mel spectrogram (`n_fft=1024`, `hop=250`) | turns sound into a 128x128 image a CNN can read |
| Per-clip loudness normalisation, scale to [-1, 1] | removes microphone-distance effects |
| Validation split **by speaker** | no voice appears in both training and validation |
| Augmentation (**training clips only**): time shift, gain, speed, pitch, noise + SpecAugment masks | more voices/conditions from few recordings |
| Class weights | compensate for class imbalance |

**In my own words (fill in):** the step that matters most for my data is ______________ because ______________________.
What I see in the augmented spectrograms below compared with the original: ______________________________________.

In [ ]:
f = random.choice(pp.list_audio(pp.TRAIN_DIR / classes[0]))
raw = pp.load_waveform(f); fitted = pp.trim_and_fit(raw)
rng = random.Random(1)
variants = [("original", pp.waveform_to_spec(fitted)[:, :, 0])]
for k in range(3):
    a = pp.spec_augment(pp.waveform_to_spec(pp.trim_and_fit(pp.augment_waveform(raw, rng))), rng)
    variants.append((f"augmented {k+1}", a[:, :, 0]))
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (t, s) in zip(axes, variants):
    ax.imshow(s, origin="lower", aspect="auto", cmap="magma"); ax.set_title(t); ax.axis("off")
plt.show()

class_names = pp.get_class_names()
train_ds, val_ds, ytr = mdl.prepare_data(class_names)
class_weights = mdl._class_weights(ytr, len(class_names))
print("classes:", class_names); print("training clips incl. augmentation:", len(ytr)); print("class weights:", class_weights)

## 5. Baseline: vanilla CNN (no pre-training, no augmentation, no regularisation tricks)

In [ ]:
from tensorflow.keras import layers
baseline = tf.keras.Sequential([
    tf.keras.Input(shape=(*pp.SPEC_SHAPE, 1)),
    layers.Conv2D(16, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(64, activation="relu"), layers.Dense(len(class_names), activation="softmax"),
], name="vanilla_cnn")
mdl.compile_model(baseline, 1e-3)
_aug = mdl.AUG_FACTOR; mdl.AUG_FACTOR = 0                      # no augmentation for the baseline
b_train, b_val, _ = mdl.prepare_data(class_names); mdl.AUG_FACTOR = _aug
h_base = baseline.fit(b_train, validation_data=b_val, epochs=max(5, EPOCHS // 2), verbose=2)

## 6. Optimised models
**A. MobileNetV2 transfer learning** (ImageNet weights, two-stage fine-tuning) and **B. a regularised CNN trained
from scratch**. Both use: dropout, L2, Adam, early stopping, ReduceLROnPlateau, class weights, augmentation.
We keep whichever scores best on the held-out test speakers.

In [ ]:
# A. MobileNetV2: stage 1 (head only) then stage 2 (fine-tune top 30 layers)
mobilenet = mdl.build_model(len(class_names)) if mdl.ARCH != "cnn" else None
if mobilenet is None:
    mdl.ARCH = "mobilenet"; mobilenet = mdl.build_model(len(class_names))
mdl.compile_model(mobilenet, 1e-3)
hA1 = mobilenet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weights, callbacks=mdl._callbacks(4), verbose=2)
mdl.unfreeze_top(mobilenet, 30); mdl.compile_model(mobilenet, 1e-5)
hA2 = mobilenet.fit(train_ds, validation_data=val_ds, epochs=FT_EPOCHS, class_weight=class_weights, callbacks=mdl._callbacks(3), verbose=2)

In [ ]:
# B. Regularised CNN from scratch
cnn = mdl.build_cnn(len(class_names)); mdl.compile_model(cnn, 1e-3)
hB = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS + 5, class_weight=class_weights, callbacks=mdl._callbacks(5), verbose=2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
histA = {k: hA1.history[k] + hA2.history[k] for k in hA1.history}
ax[0].plot(histA["accuracy"], label="train"); ax[0].plot(histA["val_accuracy"], label="val"); ax[0].set_title("MobileNetV2 accuracy")
ax[1].plot(hB.history["accuracy"], label="train"); ax[1].plot(hB.history["val_accuracy"], label="val"); ax[1].set_title("Regularised CNN accuracy")
for a in ax: a.legend(); a.set_xlabel("epoch")
plt.show()

## 7. Model testing / evaluation (accuracy, loss, precision, recall, F1, ROC-AUC, confusion matrix)

In [ ]:
# MODEL SELECTION uses the VALIDATION split (speakers held out of training). The TEST set is only used to REPORT.
Xva = np.concatenate([x.numpy() for x, _ in val_ds]); yva = np.concatenate([y.numpy() for _, y in val_ds])
files, labels = pp.list_labelled(pp.TEST_DIR, class_names)
Xte, y_true = pp.arrays_from_files(files, labels)

def full_eval(m, X, y):
    probs = m.predict(X, verbose=0); y_pred = probs.argmax(1)
    loss, acc = m.evaluate(X, y, verbose=0)
    p, r, f1, _ = precision_recall_fscore_support(y, y_pred, average="weighted", zero_division=0)
    try: auc = roc_auc_score(y, probs, multi_class="ovr", labels=list(range(len(class_names))))
    except Exception: auc = float("nan")
    return {"accuracy": acc, "loss": loss, "precision": p, "recall": r, "f1": f1, "roc_auc": auc}, y_pred

models = {"Vanilla CNN": baseline, "MobileNetV2 (transfer)": mobilenet, "Regularised CNN": cnn}
val_table = pd.DataFrame({n: full_eval(m, Xva, yva)[0] for n, m in models.items()}).T.round(4)
best_name = val_table["f1"].idxmax()
print("Validation results (used to choose the model):"); display(val_table)
print("\nBest model by VALIDATION F1:", best_name)

test_res, test_pred = {}, {}
for n, m in models.items(): test_res[n], test_pred[n] = full_eval(m, Xte, y_true)
table = pd.DataFrame(test_res).T.round(4)
print("\nTEST results on unseen speakers (reported once, not used for selection):"); table

In [ ]:
best_model, y_pred = models[best_name], test_pred[best_name]
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax[0]); ax[0].set_title("Confusion matrix (counts)")
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax[1]); ax[1].set_title("Row-normalised")
plt.tight_layout(); plt.show()

### How good is the model? (fill in after running)
* Validation F1 of the three models: baseline ______ | MobileNetV2 ______ | regularised CNN ______ -> chosen: ______
* Test accuracy / precision / recall / F1 / ROC-AUC of the chosen model: ____ / ____ / ____ / ____ / ____
* Did the optimised models beat the baseline? By how much, and which technique probably helped most
  (transfer learning, augmentation, dropout, early stopping, class weights)? ______________________________
* Training vs validation curves: is there overfitting (train accuracy far above validation)? ______________
* Confusion matrix: the two words confused most are ______ and ______ . Likely reason (similar sounds,
  similar length, speaker voice, noise): ______________________________________________________
* Which word is best recognised and which is worst (see the classification report): ___________________
* What a wrong answer means for a learner using the app, and how the confidence score can help: ____________

## 8. Prediction function

In [ ]:
metrics = mdl._save(best_model, class_names, mdl.evaluate_model(best_model, class_names), "train")
pp.log_run("train", "success", metrics["version"], metrics["accuracy"], metrics["f1"], len(ytr), note=f"notebook: {best_name}")
print("saved:", mdl.MODEL_PATH, "| version", metrics["version"], "| chose:", best_name)
prediction.reload_model()

def predict_file(path):
    r = prediction.predict_audio(open(path, "rb").read())
    return r["label"], round(r["confidence"], 3)

sample = random.sample(pp.list_audio(pp.TEST_DIR), min(8, len(pp.list_audio(pp.TEST_DIR))))
for f in sample:
    label, conf = predict_file(f)
    print(f"true: {f.parent.name:10s} predicted: {label:10s} confidence: {conf}   {'OK' if label == f.parent.name else 'WRONG'}")

## 9. Retraining demo
Simulates the UI: **new recordings from new speakers** (folder `new_recordings/`, speakers never seen in training and
never part of the test set) are uploaded, validated, saved to `data/train` and logged in the SQLite database. Retraining
then **loads the saved model as the pre-trained model** and fine-tunes it on the enlarged dataset. The test set is
untouched, so before/after scores are comparable.

In [ ]:
NEW_DIR = Path(os.getenv("NEW_DIR", "new_recordings"))
assert NEW_DIR.is_dir(), f"'{NEW_DIR}' not found: run scripts/prepare_speech_commands.py"
before = mdl.load_metrics()
n_ok = n_bad = 0
for c in class_names:
    for f in pp.list_audio(NEW_DIR / c):
        r = pp.save_uploaded_files([(f.name, f.read_bytes())], c, speaker=pp.speaker_of(f))
        n_ok += r["saved"]; n_bad += len(r["rejected"])
print(f"uploaded {n_ok} clips, rejected {n_bad} | pending (not yet used for training): {pp.pending_upload_count()}")

import sqlite3
con = sqlite3.connect(pp.DB_PATH)
print("\nUploads stored in the database:")
display(pd.read_sql("SELECT id, filename, label, speaker, uploaded_at, used_in_training FROM uploads ORDER BY id DESC LIMIT 8", con))
display(pd.read_sql("SELECT label, speaker, COUNT(*) AS clips FROM uploads GROUP BY label, speaker", con).head(12))

after = mdl.retrain(epochs=6)
pp.log_run("retrain", "success", after["version"], after["accuracy"], after["f1"], note="notebook demo")
print("\npending after retraining:", pp.pending_upload_count())
display(pd.DataFrame({"before retraining": {k: before[k] for k in ["version", "accuracy", "loss", "f1"]},
                      "after retraining": {k: after[k] for k in ["version", "accuracy", "loss", "f1"]}}))
display(pd.DataFrame(pp.list_runs()))

### Retraining results (fill in)
* New clips uploaded: ______ from ______ new speakers. Version before -> after: ____ -> ____
* Accuracy / F1 before: ______ / ______ ; after: ______ / ______ . Better, same or worse? ______
* Why (more voices? small number of new clips? too few epochs?) ______________________________________
* The two retraining triggers in my system: (1) ____________________ (2) ____________________

## 10. Conclusion (fill in)
* Best model: ______________ with test accuracy ______ and F1 ______ on speakers it had never heard.
* What worked best: ________________________ . What surprised me: ________________________
* The full pipeline: recordings -> preprocessing -> model -> evaluation -> saved `.h5` -> FastAPI -> Streamlit
  (predict, visualisations, bulk upload, retrain) -> Docker + load balancer -> cloud -> Locust tests.
* **Limitations (my own):** ______________________________________________________________
* **Next steps:** ______________________________________________________________________
* Who this helps and how it could be used in a learning app: ______________________________________